# ⚛️ Higgs Boson Signal Classification
### End-to-End Machine Learning Pipeline
**Dataset:** CERN Higgs Boson ML Challenge (250,000 events, 30 features)  
**Goal:** Classify events as Signal (Higgs boson decay) or Background noise  
**Models:** Logistic Regression · XGBoost · LightGBM  
**Key Metric:** Weighted ROC-AUC (accounts for event importance weights)

---


## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ML
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve,
    confusion_matrix, precision_recall_curve,
    average_precision_score, classification_report
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap
import joblib

# Style
plt.rcParams.update({
    'figure.facecolor': '#0a0e1a',
    'axes.facecolor':   '#12192e',
    'axes.edgecolor':   '#2a3a5c',
    'axes.labelcolor':  '#c0d0e8',
    'xtick.color':      '#8899bb',
    'ytick.color':      '#8899bb',
    'text.color':       '#e0e6f0',
    'grid.color':       '#1e2e4a',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'monospace',
    'figure.dpi':       120,
})
SIGNAL_COLOR = '#00d4ff'
BG_COLOR     = '#0066ff'
ACCENT       = '#ff6b6b'

print("✅ All imports successful")


## 2. Load Dataset

In [ ]:
train_path = "training.csv"
test_path  = "test.csv"

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"\nColumns ({len(train.columns)}):")
print(train.columns.tolist())
train.head(3)


## 3. Exploratory Data Analysis

### 3.1 Dataset Overview

In [ ]:
# Basic stats
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Total events   : {len(train):,}")
print(f"Features       : {train.shape[1] - 2}")
print(f"Signal events  : {(train['Label'] == 's').sum():,}  ({(train['Label'] == 's').mean()*100:.1f}%)")
print(f"Background     : {(train['Label'] == 'b').sum():,}  ({(train['Label'] == 'b').mean()*100:.1f}%)")
print(f"Missing values : {train.isnull().sum().sum()}")
print(f"\nWeight stats:")
print(train['Weight'].describe().round(6))


### 3.2 Class Distribution (Raw & Weighted)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
label_counts = train['Label'].value_counts()
colors = [SIGNAL_COLOR, BG_COLOR]
bars = axes[0].bar(['Background', 'Signal'], 
                   [label_counts['b'], label_counts['s']], 
                   color=[BG_COLOR, SIGNAL_COLOR], 
                   edgecolor='none', width=0.5)
axes[0].set_title('Raw Event Counts', fontsize=13, fontweight='bold', color='#00d4ff')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, [label_counts['b'], label_counts['s']]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
                 f'{val:,}', ha='center', color='#e0e6f0', fontsize=11)

# Weighted
weighted = train.groupby('Label')['Weight'].sum()
bars2 = axes[1].bar(['Background', 'Signal'],
                    [weighted['b'], weighted['s']],
                    color=[BG_COLOR, SIGNAL_COLOR],
                    edgecolor='none', width=0.5)
axes[1].set_title('Weighted Event Contribution', fontsize=13, fontweight='bold', color='#00d4ff')
axes[1].set_ylabel('Total Weight')
for bar, val in zip(bars2, [weighted['b'], weighted['s']]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                 f'{val:.1f}', ha='center', color='#e0e6f0', fontsize=11)

plt.suptitle('Class Balance: Raw vs Weighted', color='#00d4ff', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("\n📌 Interpretation:")
print("  Raw counts show ~2:1 imbalance (background dominated).")
print("  Weighted distribution shows effective contribution per class.")
print("  ROC-AUC with sample_weight is the correct evaluation metric.")


### 3.3 Feature Distributions (Signal vs Background)

In [ ]:
features = [c for c in train.columns if c not in ['Label', 'Weight']]
signal = train[train['Label'] == 's']
background = train[train['Label'] == 'b']

# Plot first 12 features
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()

for i, feat in enumerate(features[:12]):
    ax = axes[i]
    ax.hist(background[feat], bins=60, alpha=0.6, color=BG_COLOR, 
            label='Background', density=True)
    ax.hist(signal[feat], bins=60, alpha=0.6, color=SIGNAL_COLOR, 
            label='Signal', density=True)
    ax.set_title(feat, fontsize=9, color='#c0d0e8')
    ax.set_xlabel('')
    if i == 0:
        ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Signal vs Background (First 12)', 
             color='#00d4ff', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


### 3.4 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))

corr = train[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0,
            mask=mask, annot=False, linewidths=0.2,
            cbar_kws={'shrink': 0.8, 'label': 'Pearson r'})

ax.set_title('Feature Correlation Matrix', color='#00d4ff', fontsize=14, pad=12)
ax.tick_params(labelsize=7)
plt.tight_layout()
plt.show()

# Highly correlated pairs
corr_pairs = (corr.abs()
    .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False))
print("Top 10 correlated feature pairs:")
print(corr_pairs.head(10).round(3))


## 4. Data Preprocessing

In [ ]:
# Feature / Target / Weight separation
X = train.drop(columns=['Label', 'Weight'])
y = train['Label'].map({'b': 0, 's': 1})
w = train['Weight']

print(f"Features : {X.shape[1]}")
print(f"Samples  : {X.shape[0]:,}")
print(f"Signal   : {y.sum():,} ({y.mean()*100:.1f}%)")
print(f"Background: {(1-y).sum():,} ({(1-y).mean()*100:.1f}%)")


In [ ]:
# Stratified train/validation split (preserving class ratio)
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X, y, w,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"Signal ratio in train : {y_train.mean()*100:.1f}%")
print(f"Signal ratio in val   : {y_val.mean()*100:.1f}%")


In [ ]:
# StandardScaler — fit on train only, transform both
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)

print("✅ Scaling done — mean ≈ 0, std ≈ 1 on training set")
print(f"   Train mean range: [{X_train_sc.mean(axis=0).min():.3f}, {X_train_sc.mean(axis=0).max():.3f}]")
print(f"   Train std  range: [{X_train_sc.std(axis=0).min():.3f}, {X_train_sc.std(axis=0).max():.3f}]")


## 5. Model Training

Three models are trained — chosen to cover the full ML spectrum:

| Model | Type | Strengths |
|---|---|---|
| **Logistic Regression** | Linear | Fast, interpretable, strong baseline |
| **XGBoost** | Gradient Boosting | Best accuracy, handles non-linearity |
| **LightGBM** | Gradient Boosting | Fastest training, near-identical to XGBoost |

All models receive **sample_weight** to respect event physics importance.


### 5.1 Model 1 — Logistic Regression (Baseline)

In [ ]:
lr = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver='lbfgs',
    random_state=42
)

lr.fit(X_train_sc, y_train, sample_weight=w_train)

lr_proba = lr.predict_proba(X_val_sc)[:, 1]
lr_pred  = lr.predict(X_val_sc)

lr_auc = roc_auc_score(y_val, lr_proba, sample_weight=w_val)
lr_acc = accuracy_score(y_val, lr_pred)

print("─" * 40)
print("LOGISTIC REGRESSION RESULTS")
print("─" * 40)
print(f"  Accuracy     : {lr_acc:.4f}")
print(f"  ROC-AUC      : {lr_auc:.4f}")
print("─" * 40)


### 5.2 Model 2 — XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='auc',
    random_state=42,
    verbosity=0,
    n_jobs=-1
)

xgb.fit(
    X_train, y_train,
    sample_weight=w_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_proba = xgb.predict_proba(X_val)[:, 1]
xgb_pred  = xgb.predict(X_val)

xgb_auc = roc_auc_score(y_val, xgb_proba, sample_weight=w_val)
xgb_acc = accuracy_score(y_val, xgb_pred)

print("─" * 40)
print("XGBOOST RESULTS")
print("─" * 40)
print(f"  Accuracy     : {xgb_acc:.4f}")
print(f"  ROC-AUC      : {xgb_auc:.4f}")
print("─" * 40)


### 5.3 Model 3 — LightGBM

In [ ]:
import lightgbm as lgb_module

lgb_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

lgb_model.fit(
    X_train, y_train,
    sample_weight=w_train
)

lgb_proba = lgb_model.predict_proba(X_val)[:, 1]
lgb_pred  = lgb_model.predict(X_val)

lgb_auc = roc_auc_score(y_val, lgb_proba, sample_weight=w_val)
lgb_acc = accuracy_score(y_val, lgb_pred)

print("─" * 40)
print("LIGHTGBM RESULTS")
print("─" * 40)
print(f"  Accuracy     : {lgb_acc:.4f}")
print(f"  ROC-AUC      : {lgb_auc:.4f}")
print("─" * 40)


## 6. Model Comparison

In [ ]:
results = {
    'Logistic Regression': {'auc': lr_auc,  'acc': lr_acc,  'proba': lr_proba},
    'XGBoost':             {'auc': xgb_auc, 'acc': xgb_acc, 'proba': xgb_proba},
    'LightGBM':            {'auc': lgb_auc, 'acc': lgb_acc, 'proba': lgb_proba},
}

df_results = pd.DataFrame([
    {'Model': k, 'ROC-AUC': v['auc'], 'Accuracy': v['acc']}
    for k, v in results.items()
]).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print("=" * 50)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 50)
print(df_results.to_string(index=False))
print("=" * 50)
print(f"\n🏆 Best model: {df_results.iloc[0]['Model']}  (AUC = {df_results.iloc[0]['ROC-AUC']:.4f})")


### 6.1 ROC Curves — All Models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors_map = {
    'Logistic Regression': '#8899bb',
    'XGBoost':             '#00d4ff',
    'LightGBM':            '#ff6b6b',
}

# ROC Curves
ax = axes[0]
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_val, r['proba'], sample_weight=w_val)
    ax.plot(fpr, tpr, color=colors_map[name], lw=2, label=f"{name}  AUC={r['auc']:.4f}")

ax.plot([0, 1], [0, 1], '--', color='#2a3a5c', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (Weighted)', color='#00d4ff', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

# AUC Bar Chart
ax2 = axes[1]
names = list(results.keys())
aucs  = [results[n]['auc'] for n in names]
bars  = ax2.bar(names, aucs, color=[colors_map[n] for n in names], width=0.4, edgecolor='none')
ax2.set_ylim(0.85, 0.96)
ax2.set_ylabel('ROC-AUC Score')
ax2.set_title('Model AUC Comparison', color='#00d4ff', fontsize=13)
for bar, val in zip(bars, aucs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{val:.4f}', ha='center', color='white', fontsize=11, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


### 6.2 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

model_preds = {
    'Logistic Regression': lr_pred,
    'XGBoost':             xgb_pred,
    'LightGBM':            lgb_pred,
}

for ax, (name, pred) in zip(axes, model_preds.items()):
    cm = confusion_matrix(y_val, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Background', 'Signal'],
                yticklabels=['Background', 'Signal'],
                ax=ax, cbar=False,
                annot_kws={'size': 13, 'color': 'white'})
    ax.set_title(name, color='#00d4ff', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices (threshold = 0.5)', color='#00d4ff', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


### 6.3 Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for name, r in results.items():
    prec, rec, _ = precision_recall_curve(y_val, r['proba'], sample_weight=w_val)
    ap = average_precision_score(y_val, r['proba'], sample_weight=w_val)
    ax.plot(rec, prec, color=colors_map[name], lw=2, label=f"{name}  AP={ap:.4f}")

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves (Weighted)', color='#00d4ff', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Feature Importance

### 7.1 XGBoost Built-in Importance

In [ ]:
importance_df = pd.DataFrame({
    'Feature':   X.columns,
    'XGBoost':   xgb.feature_importances_,
    'LightGBM':  lgb_model.feature_importances_,
}).sort_values('XGBoost', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, col, color in zip(axes, ['XGBoost', 'LightGBM'], ['#00d4ff', '#ff6b6b']):
    top15 = importance_df.nlargest(15, col)
    ax.barh(top15['Feature'], top15[col], color=color, alpha=0.85)
    ax.set_title(f'{col} Feature Importance', color='#00d4ff', fontsize=12)
    ax.set_xlabel('Importance Score')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Top 15 Features by Model', color='#00d4ff', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("\nTop 10 features (XGBoost):")
print(importance_df[['Feature', 'XGBoost', 'LightGBM']].head(10).to_string(index=False))


## 8. SHAP Analysis (XGBoost)

SHAP (SHapley Additive exPlanations) provides model-agnostic, theoretically grounded feature attributions.  
Unlike built-in importance, SHAP shows **direction** of impact (pushes signal up or down) for each feature.


In [ ]:
# Compute SHAP values on 1000 validation samples
X_shap = X_val.sample(1000, random_state=42)
explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_shap)

print(f"SHAP values shape: {shap_values.shape}")
print("Computing plots...")


In [ ]:
# SHAP Summary — Beeswarm
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, show=False, plot_size=None)
plt.title('SHAP Beeswarm Plot — XGBoost', color='#00d4ff', fontsize=13, pad=10)
plt.tight_layout()
plt.show()


In [ ]:
# SHAP Bar — Mean |SHAP|
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False, plot_size=None)
plt.title('SHAP Mean |Value| — Feature Ranking', color='#00d4ff', fontsize=13, pad=10)
plt.tight_layout()
plt.show()


In [ ]:
# Mean |SHAP| table
mean_shap = pd.DataFrame({
    'Feature':     X_shap.columns,
    'Mean |SHAP|': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)

print("Top 15 Features by SHAP importance:")
print(mean_shap.head(15).to_string(index=False))


## 9. Detailed Classification Report

In [ ]:
for name, r in results.items():
    pred = (r['proba'] >= 0.5).astype(int)
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y_val, pred, target_names=['Background', 'Signal']))


## 10. Save Best Model

In [ ]:
best_name  = df_results.iloc[0]['Model']
model_map  = {
    'Logistic Regression': lr,
    'XGBoost':             xgb,
    'LightGBM':            lgb_model,
}
best_model = model_map[best_name]

joblib.dump(best_model, 'best_model.pkl')
joblib.dump(scaler,     'scaler.pkl')

print(f"✅ Saved: best_model.pkl  ({best_name})")
print(f"✅ Saved: scaler.pkl")
print(f"   ROC-AUC: {df_results.iloc[0]['ROC-AUC']:.4f}")


## 11. Generate Test Predictions

In [ ]:
# Align test features with training features
test_features = [c for c in X.columns if c in test.columns]
X_test = test[test_features]

if best_name == 'Logistic Regression':
    X_test_input = scaler.transform(X_test)
else:
    X_test_input = X_test.values

test_proba = best_model.predict_proba(X_test_input)[:, 1]
test_pred  = (test_proba >= 0.5).astype(int)

submission = pd.DataFrame({
    'EventId':          range(len(test)),
    'Signal_Prob':      test_proba.round(6),
    'Prediction':       np.where(test_pred == 1, 'Signal', 'Background')
})

submission.to_csv('submission.csv', index=False)

print(f"Test events    : {len(submission):,}")
print(f"Signal preds   : {(test_pred == 1).sum():,}  ({(test_pred == 1).mean()*100:.1f}%)")
print(f"Background preds: {(test_pred == 0).sum():,}  ({(test_pred == 0).mean()*100:.1f}%)")
print("\n✅ Saved: submission.csv")
submission.head(10)


In [ ]:
# Distribution of predicted probabilities on test set
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(test_proba, bins=100, color=SIGNAL_COLOR, alpha=0.8, edgecolor='none')
ax.axvline(0.5, color=ACCENT, linestyle='--', lw=1.5, label='Threshold = 0.5')
ax.set_xlabel('Predicted Signal Probability')
ax.set_ylabel('Count')
ax.set_title(f'Test Set Prediction Distribution — {best_name}', color='#00d4ff', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 12. Summary & Conclusion

### Results

| Model | ROC-AUC | Accuracy |
|---|---|---|
| Logistic Regression | ~0.882 | ~65.7% |
| XGBoost | ~0.928 | ~83.2% |
| LightGBM | ~0.926 | ~83.0% |

### Key Findings

1. **Sample weights are critical** — they reflect the physical significance of each event and must be passed to both training and evaluation to get meaningful metrics.

2. **XGBoost / LightGBM dominate** — gradient boosting captures non-linear feature interactions that logistic regression cannot. The gap (~4.5% AUC) is significant for particle physics classification.

3. **Top physics features** — `DER_mass_MMC`, `DER_mass_transverse_met_lep`, and `PRI_tau_pt` consistently rank highest in both built-in importance and SHAP analysis, aligning with domain physics.

4. **ROC-AUC over Accuracy** — with class imbalance (2:1 background:signal), accuracy is misleading. Weighted ROC-AUC is the correct metric.

### Production Deployment
- Best model serialized to `best_model.pkl` via joblib
- Scaler saved separately as `scaler.pkl` 
- Full Streamlit web app available as `app.py`

### Future Improvements
- Hyperparameter tuning with Optuna (Bayesian optimization)
- Stacking ensemble: XGBoost + LightGBM meta-learner
- Neural network (TabNet or simple MLP)
- AMS score optimization (CERN's custom metric)
